In [1]:
import pandas as pd
import numpy as np

In [2]:
clean = pd.read_csv(
    "../data/processed/occurrences_clean.csv"
)

survival_df = pd.read_csv(
    "../data/processed/survival_labels_genus.csv"
)

lazarus_df = pd.read_csv(
    "../data/processed/lazarus_taxa.csv"
)

clean["genus"] = (
    clean["accepted_name"]
    .astype(str)
    .str.split()
    .str[0]
)

In [3]:
occurrence_count = (
    clean.groupby("genus")
    .size()
    .rename("occurrence_count")
)

In [4]:
collection_count = (
    clean.groupby("genus")["collection_no"]
    .nunique()
    .rename("collection_count")
)

In [5]:
environment_breadth = (
    clean.groupby("genus")["environment"]
    .nunique()
    .rename("environment_breadth")
)

In [6]:
formation_breadth = (
    clean.groupby("genus")["formation"]
    .nunique()
    .rename("formation_breadth")
)

In [7]:
duration = (
    clean.groupby("genus")["early_interval"]
    .nunique()
    .rename("duration")
)

In [8]:
geo = clean.groupby("genus").agg(
    min_lat=("lat", "min"),
    max_lat=("lat", "max"),
    min_lng=("lng", "min"),
    max_lng=("lng", "max")
)

geo["lat_range"] = geo["max_lat"] - geo["min_lat"]
geo["lng_range"] = geo["max_lng"] - geo["min_lng"]

In [9]:
sampling_intensity = (
    clean.groupby("genus")["collection_no"]
    .count()
    .rename("sampling_intensity")
)

In [10]:
features = pd.concat([
    occurrence_count,
    collection_count,
    environment_breadth,
    formation_breadth,
    duration,
    sampling_intensity,
    geo[["lat_range", "lng_range"]]
], axis=1)

features = features.fillna(0)

print(features.shape)

features.head()

(1644, 8)


,occurrence_count,collection_count,environment_breadth,formation_breadth,duration,sampling_intensity,lat_range,lng_range
genus,,,,,,,,
Abichites,119,81,4,4,1,119,13.610001,61.050003
Abrekia,7,7,3,5,3,7,15.367334,33.908104
Abrekites,7,6,5,3,1,7,6.222103,250.452507
Abrekopsis,21,21,7,5,4,21,21.208801,246.191940
Acanthopecten,3,3,3,2,1,3,2.430001,4.175004


In [11]:
features.to_csv(
    "../data/processed/features_genus.csv"
)

In [12]:
ml_naive = survival_df.merge(
    features,
    left_on="genus",
    right_index=True,
    how="left"
)

print(ml_naive.shape)

ml_naive.head()

(2525, 11)


,genus,interval,survived,occurrence_count,collection_count,environment_breadth,formation_breadth,duration,sampling_intensity,lat_range,lng_range
0,Abichites,Changhsingian,0,119,81,4,4,1,119,13.610001,61.050003
1,Abrekia,Griesbachian,0,7,7,3,5,3,7,15.367334,33.908104
2,Abrekia,Smithian,0,7,7,3,5,3,7,15.367334,33.908104
3,Abrekia,Aegean,0,7,7,3,5,3,7,15.367334,33.908104
4,Abrekites,Smithian,0,7,6,5,3,1,7,6.222103,250.452507


In [13]:
lazarus_genera = set(
    lazarus_df["genus"]
)

In [14]:
ml_corrected = ml_naive.copy()

ml_corrected["lazarus"] = (
    ml_corrected["genus"]
    .isin(lazarus_genera)
    .astype(int)
)

In [15]:
print("Naive dataset:")
print(ml_naive.shape)

print()

print("Corrected dataset:")
print(ml_corrected.shape)

print()

print(
    "Lazarus genera:",
    ml_corrected["lazarus"].sum()
)

Naive dataset:
(2525, 11)

Corrected dataset:
(2525, 12)

Lazarus genera: 526


In [16]:
ml_naive.to_csv(
    "../data/processed/ml_dataset_naive.csv",
    index=False
)

ml_corrected.to_csv(
    "../data/processed/ml_dataset_corrected.csv",
    index=False
)

In [22]:
ml_corrected.columns.tolist()

['genus',
 'interval',
 'survived',
 'occurrence_count',
 'collection_count',
 'environment_breadth',
 'formation_breadth',
 'duration',
 'sampling_intensity',
 'lat_range',
 'lng_range',
 'lazarus']